# Build Master Monthly Dataset

This notebook merges:
- cleaned monthly WARN layoffs
- FRED macroeconomic indicators
- GDELT monthly news features

Final output:
- ../data/processed/master_monthly.csv

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
def load_fred_series(path, new_name):
    s = pd.read_csv(path)
    s.columns = [str(c).strip() for c in s.columns]

    # Handle either DATE or observation_date
    if "DATE" in s.columns:
        date_col = "DATE"
    elif "observation_date" in s.columns:
        date_col = "observation_date"
    else:
        raise ValueError(f"No recognized date column found in {path}. Columns: {list(s.columns)}")

    value_cols = [c for c in s.columns if c != date_col]
    if len(value_cols) != 1:
        raise ValueError(f"Expected 1 value column in {path}, got {value_cols}")

    value_col = value_cols[0]

    s[date_col] = pd.to_datetime(s[date_col], errors="coerce")
    s[value_col] = pd.to_numeric(s[value_col].replace(".", pd.NA), errors="coerce")
    s = s.dropna(subset=[date_col, value_col]).copy()

    s["month"] = s[date_col].dt.to_period("M").dt.to_timestamp()

    s_monthly = (
        s.groupby("month", as_index=False)[value_col]
        .mean()
        .rename(columns={value_col: new_name})
    )

    return s_monthly

In [ ]:
# Load cleaned WARN target
warn_monthly = pd.read_csv("../data/processed/warn_monthly.csv")
warn_monthly["month"] = pd.to_datetime(warn_monthly["month"])

# Load FRED
caurn = load_fred_series("../data/raw/fred/CAURN.csv", "ca_unemployment_rate")
fedfunds = load_fred_series("../data/raw/fred/FEDFUNDS.csv", "fed_funds_rate")
indeed = load_fred_series("../data/raw/fred/IHLIDXUSCA.csv", "indeed_job_postings_index")

# Load GDELT
gdelt = pd.read_csv("../data/raw/gdelt/gdelt_news_monthly.csv")
gdelt["month"] = pd.to_datetime(gdelt["month"])
gdelt["news_volume"] = pd.to_numeric(gdelt["news_volume"], errors="coerce")
gdelt["news_tone"] = pd.to_numeric(gdelt["news_tone"], errors="coerce")

In [ ]:
# Fixed month index for the exact project window
month_index = pd.DataFrame({
    "month": pd.date_range("2020-07-01", "2025-06-01", freq="MS")
})

master = (
    month_index
    .merge(warn_monthly, on="month", how="left")
    .merge(caurn, on="month", how="left")
    .merge(fedfunds, on="month", how="left")
    .merge(indeed, on="month", how="left")
    .merge(gdelt, on="month", how="left")
    .sort_values("month")
    .reset_index(drop=True)
)

# WARN is the target, so missing means zero layoffs recorded for that month
master["warn_layoffs"] = master["warn_layoffs"].fillna(0)

master.head()

In [ ]:
# Feature engineering
master["warn_layoffs_log1p"] = np.log1p(master["warn_layoffs"])
master["news_volume_log1p"] = np.log1p(master["news_volume"])

# 1-3 month lags for predictors
predictor_cols = [
    "ca_unemployment_rate",
    "fed_funds_rate",
    "indeed_job_postings_index",
    "news_volume",
    "news_volume_log1p",
    "news_tone"
]

for col in predictor_cols:
    for lag in [1, 2, 3]:
        master[f"{col}_lag{lag}"] = master[col].shift(lag)

# optional z-scores for overlay plots
plot_cols = [
    "warn_layoffs",
    "ca_unemployment_rate",
    "fed_funds_rate",
    "indeed_job_postings_index",
    "news_volume",
    "news_tone"
]

for col in plot_cols:
    std = master[col].std()
    if pd.notna(std) and std != 0:
        master[f"{col}_z"] = (master[col] - master[col].mean()) / std
    else:
        master[f"{col}_z"] = np.nan

master.head()

In [ ]:
print(caurn.head())
print(fedfunds.head())
print(indeed.head())

In [ ]:
master.shape

In [ ]:
master.isna().sum()

In [ ]:
os.makedirs("../data/processed", exist_ok=True)
master.to_csv("../data/processed/master_monthly.csv", index=False)

print("Saved: ../data/processed/master_monthly.csv")
print(master.shape)
master.isna().sum()